In [3]:
import sys
# sys.path.append("/Users/bubble/Desktop/Project/Infrasound Sensor/Layout/Code/acousticsensor")
sys.path.append("/Users/bubble/Desktop/Project/T_sensor/T_sensor")

import gdsfactory as gf
from Tools.geo_trans import round_corner
# TODO
# 1. Change one side of the cell_temp to be comparation with the width
# 2. Different with the width and same width with gap
from gdsfactory.generic_tech import get_generic_pdk
PDK = get_generic_pdk()
PDK.activate()
cell_temp = gf.Component()

/var/folders/c8/hfxc4xq534j9_8cwmvcnw5lm0000gn/T/ipykernel_37375/3037035257.py:10: DeprecationWarning: The 'gdsfactory.generic_tech' module is deprecated and will be removed in a future version. Please update your imports to use 'gdsfactory.gpdk' instead:
  from gdsfactory.gpdk import LAYER, LAYER_STACK, get_generic_pdk
Or for submodules:
  from gdsfactory.gpdk.layer_map import LAYER
  from gdsfactory.gpdk.layer_stack import LAYER_STACK
  from gdsfactory.generic_tech import get_generic_pdk


In [4]:
# bottom left
from turtle import width


L1 = [400, 900] # length of beams
w1 = 5 # width of beam
L2 = [375, 500] # length of hinges
w2 = [0.5, 0.75] # width of hinges

p = 9
grid = [None] * p
for i in range(p):
    grid[i] = gf.Component()

block = gf.Component()
count = 0
temp_T = gf.Component()
for i in range(len(L1)):
    for j in range(len(L2)):
        for k in range(len(w2)):
            row = count // 3
            col = count % 3
            # Ebeam lithography layer
            T_hinge = gf.components.rectangle(size=(w2[k], L2[j]), layer=(5, 0))
            T_beam = gf.components.rectangle(size=(L1[i], w1), layer=(5, 0))
            T_corner = round_corner(4*w2[k], w2[k], 2*w2[k], rotation=90, layer=(5, 0))
            (grid[count] << T_hinge).move((-w2[k]/2, 0))
            (grid[count] << T_beam).move((-L1[i]/2, L2[j]))
            (grid[count] << T_corner)
            (grid[count] << T_corner).dmirror_y(L2[j]/2)
            offset = 10
            frame_hinge = gf.components.rectangle(size=(w2[k]+2*offset, L2[j]-offset), layer=(6, 0))
            frame_beam = gf.components.rectangle(size=(L1[i]+2*offset, w1+2*offset), layer=(6, 0))
            (grid[count] << frame_hinge).move((-(w2[k] + 2*offset)/2, 0))
            (grid[count] << frame_beam).move((-(L1[i]+2* offset)/2, L2[j]-offset))

            # optical lithography layer
            opt_hinge = gf.components.rectangle(size=(w2[k]+offset, L2[j]-offset/2), layer=(8, 0))
            opt_beam = gf.components.rectangle(size=(L1[i]+offset, w1+offset), layer=(8, 0))
            (grid[count] << opt_hinge).move((-(w2[k] + offset)/2, 0))
            (grid[count] << opt_beam).move((-(L1[i] + offset)/2, L2[j]-offset/2))
            frame_width = L1[i] + 200
            frame_height = 2*L2[j] + w1
            opt_frame = gf.components.rectangle(size=(frame_width, frame_height), layer=(9, 0))
            (block << opt_frame).move((col * 2000, row * 2000))
            (block << grid[count]).move((frame_width/2+col * 2000, row * 2000))

            # backside etching layer
            size_x = frame_width + 743.44
            size_y = frame_height + 743.44
            diff_x = (size_x - frame_width) / 2 
            diff_y = (size_y - frame_height) / 2 
            backside = gf.components.rectangle(size=(size_x, size_y), layer=(3, 0)) 
            (block << backside).move((col * 2000 - diff_x, row * 2000 - diff_y))

            # length mark
            mark_text = gf.components.text(f"L1={L1[i]} w1={w1} l2={L2[j]} w2={w2[k]}", size=40, layer=(1, 0))
            (block << mark_text).move((col * 2000, row * 2000-200))
            if count == 1:
                (block << opt_frame).move((2 * 2000, 2 * 2000))
                (block << grid[count]).move((frame_width/2+2 * 2000, 2 * 2000))
                (block << mark_text).move((2 * 2000, 2 * 2000-200))
                (block << backside).move((2 * 2000 - diff_x, 2 * 2000 - diff_y))
            count += 1

block.show()




In [10]:
# structure for each die
# structure for 4 sides
fblock = gf.Component()
for i in range(4):
    if i == 0:
        block_ref = fblock << block
    elif i == 1:
        block_ref = fblock << block
        block_ref.move((10000, 0))
    elif i == 2:
        block_ref = fblock << block
        block_ref.move((10000, 10000))
    else:
        block_ref = fblock << block
        block_ref.move((0, 10000))

In [11]:
# order
order = gf.Component()
text_array = []
for i in range(4):
    text_array.append(gf.Component())

for i in range(4):
    T = gf.components.text(f"TB{i+1}", size=20, layer=(1, 0))
    for j in range(4):
        order_ref = text_array[i] << T
        if j == 0:
            order_ref.move((-80, 0))
        elif j == 1:
            order_ref.move((-80, 5000))
        elif j == 2:
            order_ref.move((5020, 0))
        else:
            order_ref.move((5020, 5000))
    text_array_ref = order << text_array[i]
    if i == 0:
        pass
    elif i == 1:
        text_array_ref.move((10000, 0))
    elif i == 2:
        text_array_ref.move((0, 10000))
    else:
        text_array_ref.move((10000, 10000))


In [13]:
# boolean operation
# optical lithography layer
outside = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(9, 0), layer2=(8, 0), layer=(1, 0))
cell_temp << outside

# Ebeam
T_structure = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(6, 0), layer2=(5, 0), layer=(5, 0))
cell_temp << T_structure

# add marker/order
marker = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(1, 0), layer2=(15, 0), layer=(1, 0))
cell_temp << marker
cell_temp << order

# add backside etching
backside = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(3, 0), layer2=(100, 0), layer=(3, 0))
cell_temp << backside

# frame
frame1 = gf.components.rectangle(size=(15000, 15000), layer=(20, 0))
frame2 = gf.components.rectangle(size=(20000, 20000), layer=(21, 0))
frame2_ref = cell_temp << frame2
frame2_ref.move((-2500, -2500))
cell_temp << frame1

cell_T_beam_w5 = gf.Component()
cell_ref = cell_T_beam_w5 << cell_temp
cell_ref.move((2500, -7500))
cell_T_beam_w5.show()
# cell_T_beam_w5.write_gds("mesh.gds")
# cell_T_beam_w5.plot()